In [2]:
# =========================
# CatBoost Submission Notebook
# MITSUI Commodity Prediction
# =========================

import os, gc, time, pickle
import numpy as np
import pandas as pd
from catboost import CatBoostRegressor

DATA_PATH = "/kaggle/input/competitions/mitsui-commodity-prediction-challenge"
GT_PATH = "/kaggle/input/datasets/phoenixferrari/ground-truth/test_ground_truth.csv"

VALID_OUT_PATH = "/kaggle/working/catboost_safe_valid_preds.csv"
TEST_OUT_PATH = "/kaggle/working/final_catboost_predictions.csv"
MODEL_PATH = "/kaggle/working/catboost_safe_models.pkl"

TRAIN_END_DATE_ID = 1736
VALID_START_DATE_ID = 1737
VALID_END_DATE_ID = 1826

target_cols = [f"target_{i}" for i in range(424)]
SOLUTION_NULL_FILLER = -999999

BEST_PARAMS = {
    "iterations": 150,
    "depth": 4,
    "learning_rate": 0.05,
    "l2_leaf_reg": 10,
}

N_TARGET_MODELS = 150
SAFE_CAT_WEIGHT = 1.0

/kaggle/input/datasets/phoenixferrari/the-title/catboost_fast_models.pkl


In [16]:
# =========================
# CatBoost Model
# =========================

train = pd.read_csv(f"{DATA_PATH}/train.csv")
labels = pd.read_csv(f"{DATA_PATH}/train_labels.csv")
test = pd.read_csv(f"{DATA_PATH}/test.csv")
test_gt = pd.read_csv(GT_PATH)

base_feature_cols = [c for c in train.columns if c != "is_scored"]

train_cut = train[train["date_id"] <= TRAIN_END_DATE_ID].reset_index(drop=True)
valid_cut = train[
    (train["date_id"] >= VALID_START_DATE_ID) &
    (train["date_id"] <= VALID_END_DATE_ID)
].reset_index(drop=True)

labels_train = labels[labels["date_id"] <= TRAIN_END_DATE_ID].reset_index(drop=True)
labels_valid = labels[
    (labels["date_id"] >= VALID_START_DATE_ID) &
    (labels["date_id"] <= VALID_END_DATE_ID)
].reset_index(drop=True)

X_train_base = train_cut[base_feature_cols].reset_index(drop=True)
X_valid_base = valid_cut[base_feature_cols].reset_index(drop=True)

def make_lags(label_df):
    label_df = label_df.sort_values("date_id").reset_index(drop=True)
    frames = [label_df]

    for lag in [1, 2, 3, 4]:
        lag_df = label_df[target_cols].shift(lag)
        lag_df.columns = [f"{c}_lag{lag}" for c in target_cols]
        frames.append(lag_df)

    return pd.concat(frames, axis=1).copy()

train_lagged = make_lags(labels_train)
y_train_all = train_lagged[target_cols].reset_index(drop=True)

last_known = labels_train[target_cols].tail(4).reset_index(drop=True)

# -------------------------
# Test lags from official files
# -------------------------
test_lags = test[["date_id"]].copy()

for lag in [1, 2, 3, 4]:
    lag_path = f"{DATA_PATH}/lagged_test_labels/test_labels_lag_{lag}.csv"
    lag_df = pd.read_csv(lag_path)

    if "label_date_id" in lag_df.columns:
        lag_df = lag_df.drop(columns=["label_date_id"])

    rename_map = {
        c: f"{c}_lag{lag}"
        for c in lag_df.columns
        if c.startswith("target_")
    }

    lag_df = lag_df.rename(columns=rename_map)
    keep_cols = ["date_id"] + list(rename_map.values())
    keep_cols = [c for c in keep_cols if c in lag_df.columns]

    test_lags = test_lags.merge(lag_df[keep_cols], on="date_id", how="left")

missing = {}

for lag in [1, 2, 3, 4]:
    for t in target_cols:
        col = f"{t}_lag{lag}"
        if col not in test_lags.columns:
            missing[col] = np.nan

if missing:
    test_lags = pd.concat(
        [test_lags, pd.DataFrame(missing, index=test_lags.index)],
        axis=1
    )

test_lags = test_lags.copy()

X_test_base = test.copy()

for col in base_feature_cols:
    if col not in X_test_base.columns:
        X_test_base[col] = np.nan

X_test_base = X_test_base[base_feature_cols].copy()

# -------------------------
# Metric
# -------------------------
def official_score(solution_df, pred_df):
    daily = []

    for i in range(len(pred_df)):
        y = solution_df[target_cols].iloc[i]
        p = pred_df[target_cols].iloc[i]

        valid = y.notna() & p.notna()

        if valid.sum() < 2:
            continue

        if y[valid].std(ddof=0) == 0 or p[valid].std(ddof=0) == 0:
            continue

        daily.append(np.corrcoef(
            p[valid].rank(method="average"),
            y[valid].rank(method="average")
        )[0, 1])

    daily = np.array(daily)

    if len(daily) == 0 or daily.std(ddof=0) == 0:
        return np.nan, daily

    return daily.mean() / daily.std(ddof=0), daily

# -------------------------
# Train models
# -------------------------
target_non_null_counts = y_train_all.notna().sum().sort_values(ascending=False)
trained_targets = list(target_non_null_counts.head(N_TARGET_MODELS).index)

models = {}
valid_cat_pred = {}
test_cat_pred = {}

start_total = time.time()

for i, t in enumerate(trained_targets):
    start = time.time()

    y_train = y_train_all[t]
    train_non_null = y_train.notna()

    if train_non_null.sum() < 50:
        models[t] = None
        continue

    X_train_t = X_train_base.copy()
    X_valid_t = X_valid_base.copy()
    X_test_t = X_test_base.copy()

    for lag in [1, 2, 3, 4]:
        X_train_t[f"{t}_lag{lag}"] = train_lagged[f"{t}_lag{lag}"]
        X_valid_t[f"{t}_lag{lag}"] = last_known.iloc[-lag][t]
        X_test_t[f"{t}_lag{lag}"] = test_lags[f"{t}_lag{lag}"].fillna(0.0)

    model = CatBoostRegressor(
        loss_function="RMSE",
        eval_metric="RMSE",
        random_seed=42,
        verbose=False,
        allow_writing_files=False,
        thread_count=-1,
        **BEST_PARAMS
    )

    model.fit(
        X_train_t.loc[train_non_null],
        y_train.loc[train_non_null]
    )

    models[t] = model
    valid_cat_pred[t] = model.predict(X_valid_t)
    test_cat_pred[t] = model.predict(X_test_t)

    elapsed = time.time() - start
    avg = (time.time() - start_total) / (i + 1)
    remaining = avg * (len(trained_targets) - i - 1)

    print(f"Done {t} ({i+1}/{len(trained_targets)}) | {elapsed:.1f}s | ETA={remaining/60:.1f} min")

    del X_train_t, X_valid_t, X_test_t

    if (i + 1) % 25 == 0:
        gc.collect()

# -------------------------
# Build CatBoost outputs
# -------------------------
cat_valid = pd.DataFrame(valid_cat_pred).reindex(columns=target_cols, fill_value=0.0)
cat_test = pd.DataFrame(test_cat_pred).reindex(columns=target_cols, fill_value=0.0)

# -------------------------
# Final CatBoost Prediction
# Rank normalization because metric is rank-based
# -------------------------
final_valid = cat_valid.rank(axis=1, method="average")
final_catboost_prediction = cat_test.rank(axis=1, method="average")

valid_out = pd.concat(
    [valid_cut[["date_id"]].reset_index(drop=True), final_valid.reset_index(drop=True)],
    axis=1
)

test_out = pd.concat(
    [test[["date_id"]].reset_index(drop=True), final_catboost_prediction.reset_index(drop=True)],
    axis=1
)

valid_out.to_csv(VALID_OUT_PATH, index=False)
test_out.to_csv(TEST_OUT_PATH, index=False)

with open(MODEL_PATH, "wb") as f:
    pickle.dump({
        "models": models,
        "target_cols": target_cols,
        "base_feature_cols": base_feature_cols,
        "trained_targets": trained_targets,
        "best_params": BEST_PARAMS,
        "final_model": "CatBoost-only",
        "rank_normalized": True,
        "final_prediction_file": TEST_OUT_PATH
    }, f)

print(f"\nSaved validation predictions to {VALID_OUT_PATH}")
print(f"Saved Final CatBoost Prediction to {TEST_OUT_PATH}")
print(f"Saved models to {MODEL_PATH}")

print("\nFinal validation score:")
valid_score, valid_daily = official_score(
    labels_valid[target_cols].reset_index(drop=True),
    final_valid
)
print("Validation Sharpe:", valid_score)

print("\nFinal test score:")
test_gt[target_cols] = test_gt[target_cols].replace(SOLUTION_NULL_FILLER, np.nan)

if "is_scored" in test.columns:
    scored_mask = test["is_scored"].astype(bool).values
    gt_score = test_gt.loc[scored_mask, target_cols].reset_index(drop=True)
    pred_score = final_catboost_prediction.loc[scored_mask].reset_index(drop=True)
    raw_cat_scored = cat_test.loc[scored_mask].reset_index(drop=True)
else:
    gt_score = test_gt[target_cols].reset_index(drop=True)
    pred_score = final_catboost_prediction.reset_index(drop=True)
    raw_cat_scored = cat_test.reset_index(drop=True)

test_score, test_daily = official_score(gt_score, pred_score)
raw_score, _ = official_score(gt_score, raw_cat_scored)
rank_score, _ = official_score(gt_score, pred_score)

print("Final Test Sharpe:", test_score)
print("Raw CatBoost Sharpe:", raw_score)
print("Rank CatBoost Sharpe:", rank_score)
print("Final CatBoost Prediction CSV:", TEST_OUT_PATH)

Done target_318 (1/150) | 3.7s | ETA=9.1 min
Done target_212 (2/150) | 3.7s | ETA=9.2 min
Done target_13 (3/150) | 3.7s | ETA=9.1 min
Done target_11 (4/150) | 3.8s | ETA=9.1 min
Done target_2 (5/150) | 3.7s | ETA=9.0 min
Done target_3 (6/150) | 3.6s | ETA=8.9 min
Done target_9 (7/150) | 3.6s | ETA=8.8 min
Done target_6 (8/150) | 3.7s | ETA=8.7 min
Done target_29 (9/150) | 3.8s | ETA=8.7 min
Done target_15 (10/150) | 3.6s | ETA=8.6 min
Done target_76 (11/150) | 3.6s | ETA=8.5 min
Done target_86 (12/150) | 3.8s | ETA=8.5 min
Done target_87 (13/150) | 3.6s | ETA=8.4 min
Done target_89 (14/150) | 3.7s | ETA=8.4 min
Done target_20 (15/150) | 3.6s | ETA=8.3 min
Done target_18 (16/150) | 3.6s | ETA=8.2 min
Done target_64 (17/150) | 3.6s | ETA=8.1 min
Done target_45 (18/150) | 3.6s | ETA=8.1 min
Done target_60 (19/150) | 3.8s | ETA=8.0 min
Done target_23 (20/150) | 3.6s | ETA=7.9 min
Done target_43 (21/150) | 3.7s | ETA=7.9 min
Done target_26 (22/150) | 3.7s | ETA=7.8 min
Done target_36 (23/15